**生成式 AI 使用说明**：本作业中使用生成式 AI 工具时，适用与协作相同的课程政策。和其他协作者一样，每位学生都必须独立写出自己的解答，不能直接依赖交互输出；提交内容中还应注明协作的性质。使用生成式 AI 工具实质性完成作业内容不符合本作业的精神，也会违反 [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# 将你的 Google Drive 挂载到 Colab VM。
from google.colab import drive
drive.mount('/content/drive')

# TODO：填写 Drive 中保存解压后
# 作业文件夹，例如 'cs231n/assignments/assignment1/'
FOLDERNAME = 'cs231n/assignments/assignment1/'
assert FOLDERNAME is not None, "[!] Enter the foldername."

# 现在已经挂载 Drive，下面确保
# Colab VM 的 Python 解释器可以加载
# 其中的 Python 文件。
import sys
sys.path.append('/content/drive/My Drive/{}'.format(FOLDERNAME))

# 将 CIFAR-10 数据集下载到你的 Drive
# 如果它还不存在。
%cd /content/drive/My\ Drive/$FOLDERNAME/cs231n/datasets/
!bash get_datasets.sh
%cd /content/drive/My\ Drive/$FOLDERNAME

# k-Nearest Neighbor（kNN）练习

*请完成并提交这份 worksheet，包括其中的输出以及 worksheet 外部的任何辅助代码。更多细节请参考课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。*

kNN 分类器包含两个阶段：

- 训练阶段：分类器接收训练数据，并简单地记住它们。
- 测试阶段：kNN 将每张测试图像与所有训练图像进行比较，并把最相似的 k 个训练样本的标签转移过来进行分类。
- k 的取值通过交叉验证确定。

在这个练习中，你将实现这些步骤，并理解图像分类的基本流程、交叉验证，同时熟练编写高效的向量化代码。

In [ ]:
# 运行本 notebook 的初始化代码。

import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# 这是一点 IPython magic，让 matplotlib 图像内嵌显示在 notebook 中
# 而不是显示在新窗口中。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # 设置图像的默认大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# 另一点 magic，用于让 notebook 重新加载外部 Python 模块；
# 参考 http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

In [ ]:
# 加载原始 CIFAR-10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# 清理变量，避免多次加载数据导致内存问题
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# 作为合理性检查，打印训练数据和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# 可视化 some 样本 来自 数据集.
# We show a few 样本 的 训练 images 来自每个类别.
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Subsample 数据 用于 more efficient code execution 在 这个 exercise
num_training = 5000
mask = list(range(num_training))
X_train = X_train[mask]
y_train = y_train[mask]

num_test = 500
mask = list(range(num_test))
X_test = X_test[mask]
y_test = y_test[mask]

# reshape image 数据 到 rows
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
print(X_train.shape, X_test.shape)

In [ ]:
from cs231n.classifiers import KNearestNeighbor

# Create a kNN 分类器 instance.
# Remember 该 训练 a kNN 分类器 is a noop:
# Classifier simply remembers 数据 并 does no further processing
classifier = KNearestNeighbor()
classifier.train(X_train, y_train)

现在我们希望使用 kNN 分类器对测试数据进行分类。回忆一下，这个过程可以分成两步：

1. 首先计算所有测试样本和所有训练样本之间的距离。
2. 给定这些距离后，对每个测试样本找到 k 个最近的训练样本，并让它们对标签投票。

先从计算所有训练样本和测试样本之间的距离矩阵开始。例如，如果有 **Ntr** 个训练样本和 **Nte** 个测试样本，这一步应该得到一个 **Nte x Ntr** 的矩阵，其中元素 `(i, j)` 表示第 i 个测试样本和第 j 个训练样本之间的距离。

**注意：本 notebook 要求你实现三种距离计算方式，在这些实现中不能使用 numpy 提供的 `np.linalg.norm()` 函数。**

首先打开 `cs231n/classifiers/k_nearest_neighbor.py`，实现 `compute_distances_two_loops` 函数。这个函数使用一个非常低效的双重循环，遍历所有（测试样本、训练样本）对，并逐元素计算距离矩阵。

In [ ]:
# Open cs231n/分类器s/k_nearest_邻居.py 并 implement
# compute_distances_two_loops.

# Test your 实现:
dists = classifier.compute_distances_two_loops(X_test)
print(dists.shape)

In [ ]:
# We 可以 可视化 距离矩阵: each row is a single 测试 样本 and
# its 距离 到 训练 样本
plt.imshow(dists, interpolation='none')
plt.show()

**内联问题 1**

注意距离矩阵中的结构化模式：有些行或列明显更亮。（默认配色中，黑色表示距离小，白色表示距离大。）

- 数据中的什么原因会导致明显更亮的行？
- 什么原因会导致更亮的列？

$\color{blue}{\textit 你的回答：}$ *在此填写。*

In [ ]:
# Now implement 函数 predict_标签 并 run code below:
# 我们使用 k = 1 (which is Nearest Neighbor).
y_test_pred = classifier.predict_labels(dists, k=1)

# 计算 并 print fraction 的 correctly 预测的 样本
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

你应该看到大约 `27%` 的准确率。现在尝试更大的 `k`，例如 `k = 5`：

In [ ]:
y_test_pred = classifier.predict_labels(dists, k=5)
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

你应该会看到它比 `k = 1` 的表现略好。

**内联问题 2**

我们也可以使用其他距离度量，例如 L1 距离。对于某张图像 $I_k$ 在位置 $(i,j)$ 的像素值 $p_{ij}^{(k)}$，

所有图像所有像素上的均值 $\mu$ 为：
$$\mu=\frac{1}{nhw}\sum_{k=1}^n\sum_{i=1}^{h}\sum_{j=1}^{w}p_{ij}^{(k)}$$
所有图像在每个像素位置上的逐像素均值 $\mu_{ij}$ 为：
$$\mu_{ij}=\frac{1}{n}\sum_{k=1}^np_{ij}^{(k)}.$$
整体标准差 $\sigma$ 和逐像素标准差 $\sigma_{ij}$ 也以类似方式定义。

如果最近邻分类器使用 L1 距离，下列哪些预处理步骤不会改变它的性能？请选择所有适用项。为避免歧义，训练样本和测试样本都会以相同方式预处理。

1. 减去均值 $\mu$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu$）。
2. 减去逐像素均值 $\mu_{ij}$（$\tilde{p}_{ij}^{(k)}=p_{ij}^{(k)}-\mu_{ij}$）。
3. 减去均值 $\mu$，再除以标准差 $\sigma$。
4. 减去逐像素均值 $\mu_{ij}$，再除以逐像素标准差 $\sigma_{ij}$。
5. 旋转数据的坐标轴，也就是把所有图像旋转相同角度。旋转造成的空白区域用相同像素值填充，不做插值。

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$

In [ ]:
# Now lets speed up 距离矩阵 computation by 使用 partial vectorization
# 使用 one loop. Implement 函数 compute_distances_one_loop 并 run the
# code below:
dists_one = classifier.compute_distances_one_loop(X_test)

# To ensure 该 our vectorized 实现 is correct, we 确保 该 it
# agrees 使用 naive 实现. There are many ways 到 decide whether
# two matrices are similar; one 的 simplest is Frobenius norm. In case
# you haven't seen it before, Frobenius norm 的 two matrices is square
# root 的 squared sum 的 differences 的 所有 elements; 在 other words, reshape
# matrices 到 vectors 并 计算 Euclidean 距离 between them.
difference = np.linalg.norm(dists - dists_one, ord='fro')
print('One loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# Now implement fully 向量化版本 inside compute_distances_no_loops
# and run code
dists_two = classifier.compute_distances_no_loops(X_test)

# check 该 距离矩阵 agrees 使用 one we 计算得到的 before:
difference = np.linalg.norm(dists - dists_two, ord='fro')
print('No loop difference was: %f' % (difference, ))
if difference < 0.001:
    print('Good! The distance matrices are the same')
else:
    print('Uh-oh! The distance matrices are different')

In [ ]:
# Let's compare how fast 实现s are
def time_function(f, *args):
    """
    C所有 a 函数 f 使用 args 并 return time (in seconds) 该 it took 到 execute.
    """
    import time
    tic = time.time()
    f(*args)
    toc = time.time()
    return toc - tic

two_loop_time = time_function(classifier.compute_distances_two_loops, X_test)
print('Two loop version took %f seconds' % two_loop_time)

one_loop_time = time_function(classifier.compute_distances_one_loop, X_test)
print('One loop version took %f seconds' % one_loop_time)

no_loop_time = time_function(classifier.compute_distances_no_loops, X_test)
print('No loop version took %f seconds' % no_loop_time)

# 你应该 see signifi可以tly faster performance 使用 fully vectorized 实现!

# 注意： depending on what machine you're 使用,
# you 可能 not see a speedup when you go 来自 two loops 到 one loop,
# and 可能 even see a slow-down.

### 交叉验证

我们已经实现了 k-Nearest Neighbor 分类器，但之前随意设置了 `k = 5`。现在我们将使用交叉验证来确定这个超参数的最佳取值。

In [ ]:
num_folds = 5
k_choices = [1, 3, 5, 8, 10, 12, 15, 20, 50, 100]

X_train_folds = []
y_train_folds = []
################################################################################
# TODO:                                                                        #
# Split up 训练数据 到 folds. After splitting, X_训练_folds 并    #
# y_训练_folds 应该 each be lists 的 length num_folds, 其中                #
# y_训练_folds[i] is 标签 vector 用于 点 在 X_训练_folds[i].     #
# 提示： Look up numpy 数组_split 函数.                                #
################################################################################


# A 字典 holding accuracies 用于 different 值 的 k 该 we 找到
# when running cross-验证. After running cross-验证,
# k_to_accuracies[k] 应为 a list 的 length num_folds giving different
# accuracy 值 该 we found when 使用 该 值 的 k.
k_to_accuracies = {}


################################################################################
# TODO:                                                                        #
# Perform k-fold cross 验证 到 找到 best 值 的 k. 对于每个        #
# possible 值 的 k, run k-nearest-邻居 algorithm num_folds times,   #
# 其中 在 each case you 使用 所有 but one 的 folds as 训练数据 并 #
# last fold as a 验证 set. 存储 accuracies 用于 所有 fold 并 所有     #
# 值 的 k 在 k_to_accuracies 字典.                               #
################################################################################


# Print out 计算得到的 accuracies
for k in sorted(k_to_accuracies):
    for accuracy in k_to_accuracies[k]:
        print('k = %d, accuracy = %f' % (k, accuracy))

In [ ]:
# plot raw observations
for k in k_choices:
    accuracies = k_to_accuracies[k]
    plt.scatter([k] * len(accuracies), accuracies)

# plot trend line 使用 error bars 该 correspond 到 标准差
accuracies_mean = np.array([np.mean(v) for k,v in sorted(k_to_accuracies.items())])
accuracies_std = np.array([np.std(v) for k,v in sorted(k_to_accuracies.items())])
plt.errorbar(k_choices, accuracies_mean, yerr=accuracies_std)
plt.title('Cross-validation on k')
plt.xlabel('k')
plt.ylabel('Cross-validation accuracy')
plt.show()

In [ ]:
# Based on cross-验证 结果 above, choose best 值 用于 k,
# re训练 分类器 使用 所有 训练数据, 并 测试 it on 测试
# 数据. 你应该 be able 到 get above 28% accuracy on 测试 数据.
best_k = 1

classifier = KNearestNeighbor()
classifier.train(X_train, y_train)
y_test_pred = classifier.predict(X_test, k=best_k)

# 计算并显示准确率
num_correct = np.sum(y_test_pred == y_test)
accuracy = float(num_correct) / num_test
print('Got %d / %d correct => accuracy: %f' % (num_correct, num_test, accuracy))

**内联问题 3**

在分类设置中，关于 $k$-Nearest Neighbor（$k$-NN）的下列说法哪些对所有 $k$ 都成立？请选择所有适用项。
1. k-NN 分类器的决策边界是线性的。
2. 1-NN 的训练误差总是小于或等于 5-NN 的训练误差。
3. 1-NN 的测试误差总是小于 5-NN 的测试误差。
4. 使用 k-NN 分类器对一个测试样本分类所需的时间，会随着训练集大小增长而增长。
5. 以上都不对。

$\color{blue}{\textit 你的回答：}$


$\color{blue}{\textit 你的解释：}$